# NOVA Wolf Auto Worker — FREE Colab

Специальный бесплатный сценарий для команды **«Нова, включи Colab»**.

Поток: **NOVA → Colab Free → Blender → 10s / 24fps / TRUE 360° wolf render → MP4 обратно в NOVA → runtime stop**.

Нажми **Runtime → Run all**. После появления кнопки **COPY & RETURN TO NOVA** нажми её. Дальше NOVA подхватит Connect Code, отправит уже одобренную задачу волка и после безопасного копирования MP4 на телефон попросит Colab завершить runtime.

> Google Colab Free сам решает, доступен ли T4/GPU. NOVA не обходит Google authorization и не может программно выдать себе бесплатный GPU, если Google требует ручное подтверждение.

In [ ]:
# 1) FREE GPU gate + минимальные настройки
import os, secrets, subprocess, sys, time, re, json, threading, urllib.request
from pathlib import Path

PORT = 7861
TOKEN = secrets.token_urlsafe(24)
STOP_MARKER = Path('/content/NOVA_STOP_RUNTIME')
STOP_MARKER.unlink(missing_ok=True)

probe = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(probe.stdout if probe.stdout else 'GPU пока не обнаружен.')
if probe.returncode != 0:
    raise RuntimeError('Google не выдал GPU. Runtime → Change runtime type → T4 GPU, затем Run all снова.')
print('NOVA WOLF AUTO: FREE GPU detected')


In [ ]:
# 2) Blender + FFmpeg + NOVA worker — без WanGP и без платных API
env = os.environ.copy(); env['DEBIAN_FRONTEND'] = 'noninteractive'
subprocess.run(['sudo','apt-get','update','-qq'], check=True, env=env)
subprocess.run(['sudo','apt-get','install','-y','--no-install-recommends','blender','ffmpeg','python3-venv'], check=True, env=env)
subprocess.run([sys.executable,'-m','pip','install','-q','fastapi','uvicorn','python-multipart','requests'], check=True)

REPO = Path('/content/nova-robot')
if (REPO/'.git').exists():
    subprocess.run(['git','-C',str(REPO),'fetch','origin','main'], check=True)
    subprocess.run(['git','-C',str(REPO),'checkout','main'], check=True)
    subprocess.run(['git','-C',str(REPO),'reset','--hard','origin/main'], check=True)
else:
    subprocess.run(['git','clone','--depth','1','--branch','main','https://github.com/magomedt149/nova-robot.git',str(REPO)], check=True)

required = [
    REPO/'automation/remote_gpu_worker.py',
    REPO/'automation/remote_gpu_worker_colab_auto.py',
    REPO/'blender-colab/scripts/render_wolf_cinema_auto.py',
]
for path in required:
    if not path.is_file(): raise RuntimeError(f'Missing NOVA file: {path}')
subprocess.run([sys.executable,'-m','py_compile',*[str(p) for p in required]], check=True)
subprocess.run(['blender','--version'], check=True)
subprocess.run(['ffmpeg','-version'], check=True, stdout=subprocess.DEVNULL)
print('NOVA Wolf stack: Blender + FFmpeg + worker READY')


In [ ]:
# 3) START protected worker + HTTPS Cloudflare tunnel
cloudflared = Path('/content/cloudflared')
if not cloudflared.exists():
    urllib.request.urlretrieve('https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64', cloudflared)
    cloudflared.chmod(0o755)

worker_log = Path('/content/nova_wolf_worker.log')
tunnel_log = Path('/content/nova_wolf_tunnel.log')
os.environ['NOVA_REMOTE_TOKEN'] = TOKEN
os.environ['NOVA_REMOTE_JOB_ROOT'] = '/content/NOVA_REMOTE_JOBS'
os.environ['NOVA_RUNTIME_STOP_MARKER'] = str(STOP_MARKER)

for name in ('worker','tunnel'):
    old = globals().get(name)
    if old is not None and getattr(old,'poll',lambda:0)() is None:
        try: old.terminate()
        except Exception: pass

worker_cmd = [sys.executable, str(REPO/'automation/remote_gpu_worker_colab_auto.py'), '--host','0.0.0.0','--port',str(PORT)]
worker = subprocess.Popen(worker_cmd, stdout=worker_log.open('w'), stderr=subprocess.STDOUT, env=os.environ.copy())
time.sleep(4)
if worker.poll() is not None:
    print(worker_log.read_text(errors='replace'))
    raise RuntimeError('NOVA wolf worker did not start')

tunnel = subprocess.Popen([str(cloudflared),'tunnel','--url',f'http://127.0.0.1:{PORT}','--no-autoupdate'], stdout=tunnel_log.open('w'), stderr=subprocess.STDOUT, text=True)
url = None
for _ in range(60):
    time.sleep(1)
    txt = tunnel_log.read_text(errors='replace') if tunnel_log.exists() else ''
    m = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', txt)
    if m:
        url = m.group(0); break
if not url:
    print(tunnel_log.read_text(errors='replace'))
    raise RuntimeError('Cloudflare tunnel URL not found')

import requests
health = requests.get(url+'/health', headers={'X-NOVA-Token':TOKEN}, timeout=30).json()
if not health.get('gpu',{}).get('available') or not health.get('blender') or not health.get('ffmpeg'):
    print(json.dumps(health, ensure_ascii=False, indent=2))
    raise RuntimeError('Wolf worker not ready: GPU / Blender / FFmpeg required')
print('NOVA WOLF WORKER: READY')
print(json.dumps(health, ensure_ascii=False, indent=2))


In [ ]:
# 4) Connect Code + return to NOVA + automatic runtime-stop watcher
connect_code = 'NOVA_CONNECT=' + json.dumps({'url':url,'token':TOKEN}, separators=(',',':'))
print('\n'+'='*78)
print('NOVA CONNECT CODE:', connect_code)
print('='*78)

from IPython.display import HTML, display
button_html = f'''
<button id="nova-copy" style="font-size:17px;padding:14px 18px;border-radius:12px;border:0;background:#111827;color:white;font-weight:800" onclick="(async()=>{{try{{await navigator.clipboard.writeText({json.dumps(connect_code)})}}catch(e){{}};try{{window.opener&&window.opener.postMessage({json.dumps(connect_code)},'*')}}catch(e){{}};this.innerText='Connect Code ready ✓ Returning to NOVA…';setTimeout(()=>{{try{{window.close()}}catch(e){{}};setTimeout(()=>history.back(),500)}},550)}})()">COPY & RETURN TO NOVA</button>
'''
display(HTML(button_html))
print('Нажми COPY & RETURN TO NOVA. Дальше задача волка продолжится автоматически.')

def _runtime_stop_watchdog():
    while True:
        time.sleep(2)
        if STOP_MARKER.exists():
            print('NOVA: MP4 уже перенесён. Останавливаю Colab runtime…')
            try:
                if globals().get('tunnel') is not None: tunnel.terminate()
            except Exception: pass
            try:
                if globals().get('worker') is not None: worker.terminate()
            except Exception: pass
            time.sleep(1)
            try:
                from google.colab import runtime
                runtime.unassign()
            except Exception as exc:
                print('Automatic runtime stop was not accepted by Colab:', exc)
            return

if not globals().get('nova_runtime_stop_thread') or not nova_runtime_stop_thread.is_alive():
    nova_runtime_stop_thread = threading.Thread(target=_runtime_stop_watchdog, daemon=True)
    nova_runtime_stop_thread.start()
    print('NOVA runtime-stop watcher: ACTIVE')

def _worker_watchdog():
    global worker
    while not STOP_MARKER.exists():
        time.sleep(10)
        if worker.poll() is not None and not STOP_MARKER.exists():
            log = worker_log.open('a')
            log.write('\n[NOVA WATCHDOG] restarting wolf worker...\n'); log.flush()
            worker = subprocess.Popen(worker_cmd, stdout=log, stderr=subprocess.STDOUT, env=os.environ.copy())

if not globals().get('nova_worker_watchdog_thread') or not nova_worker_watchdog_thread.is_alive():
    nova_worker_watchdog_thread = threading.Thread(target=_worker_watchdog, daemon=True)
    nova_worker_watchdog_thread.start()


## Готово

После возврата в NOVA Connect Code принимается автоматически, задача `[NOVA_WOLF_AUTO_V1]` отправляется в Blender, готовый MP4 сначала копируется из временного Colab URL в локальный Blob на телефоне, и только после успешного переноса NOVA вызывает `/runtime/shutdown`.

Если локальное копирование MP4 не удалось, NOVA **не** завершает runtime, чтобы не потерять результат. Платные API и WanGP в этом notebook не используются.